 In this lesson, we'll examine all the features, selecting five to create clusters with. After we build our model and choose an appropriate number of clusters, we'll learn how to visualize multi-dimensional clusters in a 2D scatter plot using something called principal component analysis (PCA).

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving SCFP2019.csv to SCFP2019.csv


Prepare Data
Import

Rewrite your wrangle function from the last lesson so that it returns a DataFrame of households whose net worth is less than $2 million and that have been turned down for credit or feared being denied credit in the past 5 years (see "TURNFEAR").

In [ ]:
def wrangle(filepath):
  df=pd.read_csv(filepath)
  mask=(df["TURNFEAR"]==1)&(df["TURNFEAR"]<2e6)
  df=df[mask]
  return df

In [ ]:
import pandas as pd
df=wrangle("SCFP2019.csv")
print("df type:",type(df))
print("df shape:",df.shape)
print(df.head())


df type: <class 'pandas.core.frame.DataFrame'>
df shape: (4623, 362)
   YY1  Y1          WGT  HHSEX  AGE  AGECL  EDUC  EDCL  MARRIED  KIDS  ...  \
5    2  21  3790.476607      1   50      3     8     2        1     3  ...   
6    2  22  3798.868505      1   50      3     8     2        1     3  ...   
7    2  23  3799.468393      1   50      3     8     2        1     3  ...   
8    2  24  3788.076005      1   50      3     8     2        1     3  ...   
9    2  25  3793.066589      1   50      3     8     2        1     3  ...   

   NWCAT  INCCAT  ASSETCAT  NINCCAT  NINC2CAT  NWPCTLECAT  INCPCTLECAT  \
5      1       2         1        2         1           1            4   
6      1       2         1        2         1           1            4   
7      1       2         1        2         1           1            4   
8      1       2         1        2         1           1            4   
9      1       2         1        2         1           1            4   

   NINCPCTLECAT  

                      EXPLORE
    In this lesson, we want to make clusters using more than two features,
     but which of the 351 features should we choose? Often times,
     this decision will be made for you. For example, a stakeholder could give you a list of the features that are most important to them.
     If you don't have that limitation, though,
     another way to choose the best features for clustering is to determine which numerical features have the largest

>



Variance
Variance (which is sometimes called volatility in finance), is a measurement of how spread out the points in a dataset are around the mean. It's calculated by first summing up the squared different between each data point and mean, then dividing by the number of data points minus

In [ ]:
#Calculate the variance for all the features in df, and create a Series top_ten_var with the 10 features with the largest variance.
top_ten_var=df.var().sort_values().tail(10)
print("top_ten_var typr:",type(top_ten_var))
print("top_ten_var:",top_ten_var)


top_ten_var typr: <class 'pandas.core.series.Series'>
top_ten_var: FIN          1.790325e+13
NONACTBUS    3.785699e+13
ACTBUS       1.875897e+14
KGBUS        3.025702e+14
KGTOTAL      3.318717e+14
BUS          3.701502e+14
NFIN         4.159772e+14
NETWORTH     4.914040e+14
ASSET        5.095533e+14
NHNFIN       6.070985e+14
dtype: float64


 Use plotly express to create a horizontal bar chart of top_ten_var. Be sure to label your x-axis "Variance", the y-axis "Feature", and use the title "SCF: High Variance Features".

In [ ]:
import plotly.express as px
fig=px.bar(
    x=top_ten_var,
    y=top_ten_var.index,
    title="SCF: High Variance Features"
)
fig.update_layout(
    xaxis_title="Variance",
    yaxis_title="Feature"
)
fig.show();

Boxplots
A boxplot is a graph that shows the minimum, first quartile, median, third quartile, and the maximum values in a dataset. Boxplots are useful, as they provide a visual summary of the data enabling researchers to quickly identify mean values, the dispersion of the data set, and signs of skewness

Use plotly express to create a horizontal boxplot of "NHNFIN" to determine if the values are skewed. Be sure to label the x-axis "Value [$]", and use the title "Distribution of Non-home, Non-Financial Assets".

In [ ]:
fig=px.box(
    data_frame=df,
    x="NHNFIN",
    title="Distribution of Non-home, Non-Financial Assets"
)
fig.update_layout(
    xaxis_title="Value [$]"
)
fig.show();

 The dataset is massively right-skewed because of the huge outliers on the right side of the distribution. Even though we already excluded households with a high net worth with our wrangle function, the variance is still being distorted by some extreme outliers.

The best way to deal with this is to look at the trimmed variance, where we remove extreme values before calculating variance. We can do this using the trimmed_variance function from the SciPy library.

 Trimmed variance is the variance calculated excluding the largest and the smallest data points.

Calculate the trimmed variance for the features in df. Your calculations should not include the top and bottom 10% of observations. Then create a Series top_ten_trim_var with the 10 features with the largest variance.

In [ ]:
from scipy.stats.mstats import trimmed_var
top_ten_trim_var=df.apply(trimmed_var,limits=(0.1,0.1)).sort_values().tail(10)
print("trimmed_var type:",type(top_ten_trim_var))
print(top_ten_trim_var.head())


trimmed_var type: <class 'pandas.core.series.Series'>
KGTOTAL    1.073830e+09
HOMEEQ     1.965624e+09
NH_MORT    3.028119e+09
MRTHEL     3.213184e+09
PLOAN1     3.349231e+09
dtype: float64


 Use plotly express to create a horizontal bar chart of top_ten_trim_var. Be sure to label your x-axis "Trimmed Variance", the y-axis "Feature", and use the title "SCF: High Variance Features"

In [ ]:
fig = px.bar(
    x=top_ten_trim_var,
    y=top_ten_trim_var.index,
    title="SCF: High Variance Features",
    orientation="h"
)
fig.update_layout(
    xaxis_title="Trimmed Variance",
    yaxis_title="Feature"
)
fig.show();

There are three things to notice in this plot. First, the variances have decreased a lot. In our previous chart, the x-axis went up to $80 billion; this one goes up to $12 billion. Second, the top 10 features have changed a bit. All the features relating to business ownership ("...BUS") are gone. Finally, we can see that there are big differences in variance from feature to feature. For example, the variance for "WAGEINC" is around than $500 million, while the variance for "ASSET" is nearly $12 billion. In other words, these features have completely different scales. This is something that we'll need to address before we can make good clusters

In [ ]:
#Generate a list high_var_cols with the column names of the five features with the highest trimmed variance.
high_var_cols=top_ten_trim_var.tail(5).index.to_list()
print("high_var_cols type:",type(high_var_cols))
print("high_var_cols len:",len(high_var_cols))
print(high_var_cols)

high_var_cols type: <class 'list'>
high_var_cols len: 5
['DEBT', 'NETWORTH', 'HOUSES', 'NFIN', 'ASSET']


In [ ]:
#: Create the feature matrix X. It should contain the five columns in high_var_cols.
X=df[high_var_cols]
print("X type:",type(X))
print("X shape:",X.shape)
print(X.head())

X type: <class 'pandas.core.frame.DataFrame'>
X shape: (4623, 5)
           DEBT      NETWORTH  HOUSES          NFIN         ASSET
5  15477.033113  -8512.368212     0.0   4947.576159   6964.664901
6  15984.476821  -5975.149669     0.0   7992.238411  10009.327152
7  19409.721854 -10294.764240     0.0   7104.211920   9114.957616
8  17887.390728  -3184.209272     0.0  12686.092715  14703.181457
9  19536.582781  -7250.101987     0.0  10275.735099  12286.480795


    iterate
During our EDA, we saw that we had a scale issue among our features. That issue can make it harder to cluster the data, so we'll need to fix that to help our analysis along. One strategy we can use is standardization, a statistical method for putting all the variables in a dataset on the same scale. Let's explore how that works here. Later, we'll incorporate it into our model pipeline.

In [ ]:
#Create a DataFrame X_summary with the mean and standard deviation for all the features in X.
X_summary=X.aggregate({"mean","std"})
print("X_summary type:",type(X_summary))
print("X_summary shape:",X_summary.shape)
X_summary.head()

X_summary type: <class 'pandas.core.frame.DataFrame'>
X_summary shape: (2, 5)


,DEBT,NETWORTH,HOUSES,NFIN,ASSET
mean,161042.142727,1.799476e+06,2.420483e+05,1.571507e+06,1.960519e+06
std,883360.351301,2.216763e+07,1.732288e+06,2.039552e+07,2.257329e+07


Create a StandardScaler transformer, use it to transform the data in X, and then put the transformed data into a DataFrame named X_scaled.

In [ ]:
from sklearn.preprocessing import StandardScaler
#intiate the standard scaler
ss=StandardScaler()

In [ ]:
#fit and transform

X_scaled_data=ss.fit_transform(X)

In [ ]:
#put the X_scaled_data into a data frame
X_scaled=pd.DataFrame(X_scaled_data)
print("X_scaled type:",type(X_scaled))
print("X_scaled shape:",X_scaled.shape)
print(X_scaled.head())

X_scaled type: <class 'pandas.core.frame.DataFrame'>
X_scaled shape: (4623, 5)
          0         1         2         3         4
0 -0.164803 -0.081569 -0.139743 -0.076817 -0.086552
1 -0.164229 -0.081454 -0.139743 -0.076668 -0.086417
2 -0.160351 -0.081649 -0.139743 -0.076712 -0.086457
3 -0.162075 -0.081328 -0.139743 -0.076438 -0.086209
4 -0.160207 -0.081512 -0.139743 -0.076556 -0.086316


As you can see, all five of the features use the same scale now. But just to make sure, let's take a look at their mean and standard deviation.

Create a DataFrame X_scaled_summary with the mean and standard deviation for all the features in X_scaled.

In [ ]:
X_scaled_summary=X_scaled.aggregate(["mean","std"]).astype(int)
print("X_scaled_summary type:",type(X_scaled_summary))
print("X_scaled_summary shape:",X_scaled_summary.shape)
print(X_scaled_summary.head())

X_scaled_summary type: <class 'pandas.core.frame.DataFrame'>
X_scaled_summary shape: (2, 5)
      0  1  2  3  4
mean  0  0  0  0  0
std   1  1  1  1  1


And that's what it should look like. Remember, standardization takes all the features and scales them so that they all have a mean of 0 and a standard deviation of 1.

Use a for loop to build and train a K-Means model where n_clusters ranges from 2 to 12 (inclusive). Your model should include a StandardScaler. Each time a model is trained, calculate the inertia and add it to the list inertia_errors, then calculate the silhouette score and add it to the list silhouette_scores

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.pipeline import make_pipeline

In [ ]:
n_clusters = range(2, 13)
inertia_errors = []
silhouette_scores = []

for n in n_clusters:
    # build a model
    model = make_pipeline(
        StandardScaler(),
        KMeans(n_clusters=n, random_state=42)
    )
    # fit the model
    model.fit(X)
    # calculate the inertia
    inertia_errors.append(model.named_steps["kmeans"].inertia_)
    # calculate the silhouette scores
    silhouette_scores.append(silhouette_score(X, model.named_steps["kmeans"].labels_))

print("inertia_errors type:", type(inertia_errors))
print("inertia_errors len:", len(inertia_errors))
print("Inertia:", inertia_errors)

print()

print("silhouette_scores type:", type(silhouette_scores))
print("silhouette_scores len:", len(silhouette_scores))
print("Silhouette Scores:", silhouette_scores)

inertia_errors type: <class 'list'>
inertia_errors len: 11
Inertia: [7789.322200552528, 3498.1943549165303, 2193.638256472056, 1145.2926867995425, 768.6610507930807, 620.9715453140548, 520.6640045587238, 435.99172741607634, 416.08412250811773, 367.91503406963767, 294.92120771077214]

silhouette_scores type: <class 'list'>
silhouette_scores len: 11
Silhouette Scores: [np.float64(0.99264951905634), np.float64(0.9859234178309199), np.float64(0.9874886122015244), np.float64(0.9455859154340251), np.float64(0.9553758526221132), np.float64(0.8312979086487792), np.float64(0.5686966138731001), np.float64(0.5693798250451304), np.float64(0.6240957952929138), np.float64(0.5626136282005285), np.float64(0.539222899912031)]


: Use plotly express to create a line plot that shows the values of silhouette_scores as a function of n_clusters. Be sure to label your x-axis "Number of Clusters", your y-axis "Silhouette Score", and use the title "K-Means Model: Silhouette Score vs Number of Clusters"

In [ ]:
fig=px.line(
    x=n_clusters,
    y=silhouette_scores,
    title="K-Means Model: Silhouette Score vs Number of Clusters"
)
fig.update_layout(
    xaxis_title="Number of Clusters",
    yaxis_title= "Silhouette Score"
)
fig.show();


This one's a little less straightforward, but we can see that the best silhouette scores occur when there are 3 or 4 clusters.

Build and train a new k-means model named final_model. Use the information you gained from the two plots above to set an appropriate value for the n_clusters argument. Once you've built and trained your model, run Check Activity to verify your results.

In [ ]:
final_model=make_pipeline(
    StandardScaler(),
    KMeans(
        n_clusters=4,
        random_state=42
    )
)

In [ ]:
final_model.fit(X)

Pipeline(steps=[('standardscaler', StandardScaler()),
                ('kmeans', KMeans(n_clusters=4, random_state=42))])

Communicate

Extract the labels that your final_model created during training and assign them to the variable labels.

In [ ]:
labels=final_model.named_steps["kmeans"].labels_
print("labels type:",type(labels))
print("labels shape:",labels.shape)
print(labels[:5])

labels type: <class 'numpy.ndarray'>
labels shape: (4623,)
[0 0 0 0 0]


 Create a DataFrame xgb that contains the mean values of the features in X for each of the clusters in your final_model.

In [ ]:
xgb=X.groupby(labels).mean()
print("xgb type:",type(xgb))
print("xgb shape:",xgb.shape)
print(xgb.head())

xgb type: <class 'pandas.core.frame.DataFrame'>
xgb shape: (4, 5)
           DEBT      NETWORTH        HOUSES          NFIN         ASSET
0  1.288171e+05  5.542022e+05  1.719906e+05  4.773241e+05  6.830193e+05
1  6.459251e+06  6.069697e+08  1.917376e+07  5.659791e+08  6.134289e+08
2  2.332224e+07  1.566461e+08  4.374672e+07  1.741741e+08  1.799684e+08
3  2.193003e+05  1.301786e+08  9.049413e+05  9.130484e+07  1.303979e+08


 Use plotly express to create a side-by-side bar chart from xgb that shows the mean of the features in X for each of the clusters in your final_model. Be sure to label the x-axis "Cluster", the y-axis "Value [$]", and use the title "Mean Household Finances by Cluster".

In [ ]:
# Create side-by-side bar chart of `xgb`
fig = px.bar(
    xgb,
    barmode="group",
    title="Mean Household Finances by Cluster"

)
fig.update_layout(
    xaxis_title="Cluster",
    yaxis_title="Value [$]"
)
fig.show()

Remember that our clusters are based partially on NETWORTH, which means that the households in the 0 cluster have the smallest net worth, and the households in the 2 cluster have the highest. Based on that, there are some interesting things to unpack here.

First, take a look at the DEBT variable. You might think that it would scale as net worth increases, but it doesn't. The lowest amount of debt is carried by the households in cluster 2, even though the value of their houses (shown in green) is roughly the same. You can't really tell from this data what's going on, but one possibility might be that the people in cluster 2 have enough money to pay down their debts, but not quite enough money to leverage what they have into additional debts. The people in cluster 3, by contrast, might not need to worry about carrying debt because their net worth is so high.

Finally, since we started out this project looking at home values, take a look at the relationship between DEBT and HOUSES. The value of the debt for the people in cluster 0 is higher than the value of their houses, suggesting that most of the debt being carried by those people is tied up in their mortgages — if they own a home at all. Contrast that with the other three clusters: the value of everyone else's debt is lower than the value of their homes.

So all that's pretty interesting, but it's different from what we did last time, right? At this point in the last lesson, we made a scatter plot. This was a straightforward task because we only worked with two features, so we could plot the data points in two dimensions. But now X has five dimensions! How can we plot this to give stakeholders a sense of our clusters?

Since we're working with a computer screen, we don't have much of a choice about the number of dimensions we can use: it's got to be two. So, if we're going to do anything like the scatter plot we made before, we'll need to take our 5-dimensional data and change it into something we can look at in 2 dimensions.

```
# This is formatted as code
```



Create a PCA transformer, use it to reduce the dimensionality of the data in X to 2, and then put the transformed data into a DataFrame named X_pca. The columns of X_pca should be named "PC1" and "PC2".

In [ ]:
from sklearn.decomposition import PCA

# Instantiate transformer
pca = PCA(n_components=2, random_state=42)

# Transform `X`
X_t = pca.fit_transform(X)

# Put `X_t` into DataFrame
X_pca = pd.DataFrame(X_t, columns=["PC1", "PC2"])

print("X_pca type:", type(X_pca))
print("X_pca shape:", X_pca.shape)
X_pca.head()

X_pca type: <class 'pandas.core.frame.DataFrame'>
X_pca shape: (4623, 2)


,PC1,PC2
0,-3.092177e+06,81521.789572
1,-3.087205e+06,81988.169377
2,-3.090730e+06,83641.612267
3,-3.080188e+06,83266.856634
4,-3.085318e+06,83926.782748


 Use plotly express to create a scatter plot of X_pca using seaborn. Be sure to color the data points using the labels generated by your final_model. Label the x-axis "PC1", the y-axis "PC2", and use the title "PCA Representation of Clusters".

In [ ]:
# Create scatter plot of `PC2` vs `PC1`
fig = px.scatter(
    data_frame=X_pca,
    x="PC1",
    y="PC2",
    color=labels.astype(str),
    title="PCA Representation of Clusters"

)
fig.update_layout(
    xaxis_title="Cluster",
    yaxis_title="Value [$]"
)

fig.show()

 we built a model based on the highest-variance features in our dataset and created several visualizations to communicate our results. In this lesson, we're going to combine all of these elements into a dynamic web application that will allow users to choose their own features, build a model, and evaluate its performance through a graphic user interface. In other words, you'll create a tool that will allow anyone to build a model without code.

Build Dashboard

Application Layout

In [ ]:
!pip install dash

#: Instantiate a Dash application and assign it to the variable name app
from dash import Dash
from dash import Input, Output, dcc, html

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 59.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [dash]


In [ ]:
app=Dash(__name__)
print("app type:",type(app))

app type: <class 'dash.dash.Dash'>


Start building the layout of your app by creating a Div object that has two child objects: an H1 header that reads "Survey of Consumer Finances" and an H2 header that reads "High Variance Features".

In [ ]:
app.layout = html.Div(
    [
       # application title
       html.H1("Survey of Consumer Finances"),
       # bar chart element
       html.H2("High Variance Features "),
       # Add a Graph object to your application's layout. Be sure to give it the id "bar-chart" (bar chart presentation)
       dcc.Graph(figure=serve_bar_chart(), id="bar-chart"),
       # add radio button
       dcc.RadioItems(
           options=[
               {"label": "trimmed", "value": True},
               {"label": "untrimmed", "value": False}
           ],
           value=True,
           id="trimmed_button"
       ),
       #Add two text objects to your application's layout: an H2 header that reads "K-means Clustering"
       #and an H3 header that reads "Number of Clusters (k)"
       html.H2("K_means Clustering"),
       html.H3("Number of Clusters(k)"),
       #Add a slider to your application's layout. It should range from 2 to 12. Be sure to give it the id "k-slider"
       dcc.Slider(min=2,max=12,value=2,id="k-slider"),
       #Add a Div object to your applications layout. Be sure to give it the id "metrics"
       html.Div(id="metreics"),
       #Add a Graph object to your application's layout. Be sure to give it the id "pca-scatter"
       dcc.Graph("pca-scatter")
    ]
)

Create a get_high_var_features function that returns the five highest-variance features in a DataFrame. Use the docstring for guidance

In [ ]:
def get_high_var_features(trimmed=True, return_feat_names=True):
    """Returns the five highest-variance features of ``df``.

    Parameters
    ----------
    trimmed : bool, default=True
        If ``True``, calculates trimmed variance, removing bottom and top 10%
        of observations.

    return_feat_names : bool, default=False
        If ``True``, returns feature names as a ``list``. If ``False``
        returns ``Series``, where index is feature names and values are
        variances.
    """
    # calculate variance
    if trimmed:
        top_five_features = df.apply(trimmed_var, limits=(0.1, 0.1)).sort_values().tail(5)
    else:
        top_five_features = df.var().sort_values().tail(5)
    # extracts names
    if return_feat_names:
        return top_five_features.index.to_list()

    return top_five_features

In [ ]:
get_high_var_features(trimmed=True, return_feat_names=True)

['DEBT', 'NETWORTH', 'HOUSES', 'NFIN', 'ASSET']

Create a serve_bar_chart function that returns a plotly express bar chart of the five highest-variance features. You should use get_high_var_features as a helper function. Follow the docstring for guidance.

In [ ]:
@app.callback(
    Output("bar-chart", "figure"),
    Input("trimmed_button", "value")
)
def serve_bar_chart(trimmed=True):
    """Returns a horizontal bar chart of five highest-variance features.

    Parameters
    ----------
    trimmed : bool, default=True
        If ``True``, calculates trimmed variance, removing bottom and top 10%
        of observations.
    """
    # get the features
    high_var_features = get_high_var_features(
        trimmed=trimmed,
        return_feat_names=False
    )
    # build bar
    fig = px.bar(
        x=high_var_features,
        y=high_var_features.index,
        orientation="h"
    )
    fig.update_layout(
        xaxis_title="Variance",
        yaxis_title="Feature"
    )
    return fig

In [ ]:
serve_bar_chart(trimmed=False)

Create a get_model_metrics function that builds, trains, and evaluates KMeans model. Use the docstring for guidance. Note that, like the model you made in the last lesson, your model here should be a pipeline that includes a StandardScaler

In [ ]:
def get_model_metrics(trimmed=True, k=2, return_metrics=False):
    """Build a KMeans model based on five highest-variance features in df.

    Parameters
    ----------
    trimmed : bool, default=True
        If True, calculates trimmed variance, removing the bottom and
        top 10% of observations.

    k : int, default=3
        Number of clusters to create.

    return_metrics : bool, default=False
        If False, returns the fitted KMeans model.
        If True, returns a dictionary containing inertia and
        silhouette score.

    Returns
    -----
    model : sklearn.pipeline.Pipeline
        Fitted KMeans pipeline if return_metrics=False.

    metrics : dict
        Dictionary containing inertia and silhouette score
        if return_metrics=True.
    """
    # Get the features
    features = get_high_var_features(
        trimmed=trimmed,
        return_feat_names=True
    )
    # get the feature matrix
    X = df[features]
    # build the model
    model = make_pipeline(
        StandardScaler(),
        KMeans(n_clusters=k, random_state=42)
    )
    model.fit(X)

    if return_metrics:
        # calculate the inertia
        i = model.named_steps["kmeans"].inertia_
        # calculate the silhouette score
        ss = silhouette_score(X, model.named_steps["kmeans"].labels_)
        # put the results into a dictionary
        metrics = {
            "inertia": i,
            "silhouette_score": round(ss, 3)
        }
        # return the dictionary to the user
        return metrics

    return model

In [ ]:
get_model_metrics(trimmed=True,k=2,return_metrics=False)

Pipeline(steps=[('standardscaler', StandardScaler()),
                ('kmeans', KMeans(n_clusters=2, random_state=42))])

Create a serve_metrics function. It should use your get_model_metrics to build and get the metrics for a model, and then return two objects: An H3 header with the model's inertia and another H3 header with the silhouette score.

In [ ]:
#Add a callback decorator to your serve_metrics function. The callback inputs should be the values returned by "trim-button" and "k-slider",
#and the output should be directed to "metrics".
@app.callback(
    Output("metrics","children"),
    Input("trim-button","value"),
    Input("k-slider","value")
)
def serve_metrics(trimmed=True, k=2):
    """Returns list of ``H3`` elements containing inertia and silhouette score
    for ``KMeans`` model.

    Parameters
    ----------
    trimmed : bool, default=True
        If ``True``, calculates trimmed variance, removing bottom and top 10%
        of observations.

    k : int, default=2
        Number of clusters.
    """
    # get the metrics
    metrics = get_model_metrics(trimmed=trimmed, k=k, return_metrics=True)
    # agg the metrics to html
    text = [
        html.H3(f"Inertia: {metrics['inertia']}"),
        html.H3(f"Silhouette Score: {metrics['silhouette_score']}")
    ]

    return text

In [ ]:
 serve_metrics(trimmed=True,k=2)

[H3('Inertia: 7789.322200552528'), H3('Silhouette Score: 0.993')]

PCA Scatter Plot

Create a function get_pca_labels that subsets a DataFrame to its five highest-variance features, reduces those features to two dimensions using PCA, and returns a new DataFrame with three columns: "PC1", "PC2", and "labels". This last column should be the labels determined by a KMeans model. Your function should you get_high_var_features and get_model_metrics as helpers. Refer to the docstring for guidance

In [ ]:

def get_pca_labels(trimmed=True, k=2):
    """Return a DataFrame containing PCA components and KMeans labels.

    Parameters
    ----------
    trimmed : bool, default=True
        If True, calculates trimmed variance, removing the bottom and
        top 10% of observations.

    k : int, default=2
        Number of clusters to create.

    Returns
    -------
    X_pca : pandas.DataFrame
        DataFrame containing two principal components (PC1 and PC2)
        and their corresponding KMeans cluster labels.
    """
    # 1. Get the 5 highest-variance features
    features = get_high_var_features(trimmed=trimmed, return_feat_names=True)
    X = df[features]

    # 2. Reduce the features to 2 dimensions using PCA
    pca = PCA(n_components=2, random_state=42)
    X_t = pca.fit_transform(X)

    # 3. Create a new DataFrame with PC1 and PC2
    X_pca = pd.DataFrame(X_t, columns=["PC1", "PC2"])

    # 4. Get the fitted KMeans pipeline model
    model = get_model_metrics(trimmed=trimmed, k=k, return_metrics=False)

    # 5. Extract and add the KMeans cluster labels to the DataFrame
    X_pca["labels"] = model.named_steps["kmeans"].labels_.astype(str)

    # 6. Sort the DataFrame by cluster labels
    X_pca = X_pca.sort_values("labels")

    return X_pca

In [ ]:
get_pca_labels(trimmed=True, k=2)

,PC1,PC2,labels
0,-3.092177e+06,8.152179e+04,0
3081,-2.999586e+06,1.929146e+04,0
3080,-2.995978e+06,1.699028e+04,0
3079,-2.102388e+06,2.945070e+05,0
3078,-2.268594e+06,3.859005e+05,0
...,...,...,...
231,2.782182e+08,2.448364e+07,1
232,2.715615e+08,2.416284e+07,1
234,3.032351e+08,2.524325e+07,1
3323,9.865588e+08,7.520386e+06,1


Create a function serve_scatter_plot that creates a 2D scatter plot of the data used to train a KMeans model, along with color-coded clusters. Use get_pca_labels as a helper. Refer to the docstring for guidance.

In [ ]:
#Add a callback decorator to your serve_metrics function. The callback inputs should be the values returned by "trim-button" and "k-slider",
#and the output should be directed to "metrics".
@app.callback(
    Output("metrics","children"),
    Input("trim-button","value"),
    Input("k-slider","value")
)
def serve_scatter_plot(trimmed=True, k=2):
    """Build 2D scatter plot of ``df`` with ``KMeans`` labels.

    Parameters
    ----------
    trimmed : bool, default=True
        If ``True``, calculates trimmed variance, removing bottom and top 10%
        of observations.


    k : int, default=2
        Number of clusters.
    """
    fig = px.scatter(
        data_frame=get_pca_labels(trimmed=trimmed, k=k),
        x="PC1",
        y="PC2",
        color="labels",
        title="PCA Representation of Clusters"
    )
    fig.update_layout(
        xaxis_title="PC1",
        yaxis_title="PC2"
    )
    return fig

In [ ]:
serve_scatter_plot(trimmed=True, k=5)

Application Deployment

In [ ]:
# create a public server
from google.colab import output
port = 9050
public_url = output.eval_js(f"google.colab.kernel.proxyPort({port})")
print(f"Your public Dash app URL is: {public_url}")
#This code starts your Dash web application and makes it accessible through a network connection
app.run(host="0.0.0.0", port=9050, debug=False)

Your public Dash app URL is: https://9050-m-s-euw4b7-26rcyqfoxw8kt-b.europe-west4.prod.r.colab.dev


<IPython.core.display.Javascript object>

### 🚀 Dash Web Application Public Link

Click the link below to open your running Dash application in a new browser tab:

👉 **[Open Dash Web Application](https://9050-m-s-use1b13-i226i2cqj1hn-b.us-east1.prod.r.colab.dev)**

*(Note: Make sure the server cell is active and running while you visit this link.)*

This code starts your Dash web application and makes it accessible through a network connection